In [1]:
import pandas as pd
import numpy as np

In [4]:
# ── Load raw files (same as training pipeline) ────────────────────────────────
billings = pd.read_csv("dataset/raw/billings.csv")
rc       = pd.read_csv("dataset/preprocessed/renewal_calls_preprocessed.csv")
cc       = pd.read_csv("dataset/preprocessed/cc_calls_preprocessed.csv")
emails   = pd.read_csv("dataset/preprocessed/emails_preprocessed.csv")
 
print(f"Billings raw : {billings.shape}")
print(f"Open rows    : {(billings['Prospect_Outcome'] == 'Open').sum()}")

C:\Users\pragi\AppData\Local\Temp\ipykernel_9244\3500397624.py:2: DtypeWarning: Columns (15,16,19,52,53) have mixed types. Specify dtype option on import or set low_memory=False.
  billings = pd.read_csv("dataset/raw/billings.csv")


Billings raw : (122082, 59)
Open rows    : 8188


In [5]:
# Step 1 — Drop useless columns
cols_to_drop = [
    'Last_Years_Date_Paid', 'Last_Payment_Score', 'Last_Discount_Score',
    'Billings_Contractor', 'DateTime_Out', 'Current_Anchor_List',
]
billings.drop(columns=[c for c in cols_to_drop if c in billings.columns],
              inplace=True)
 
# Step 2 — Fix date columns
date_cols = ['Renewal_Month', 'Proforma_Date', 'Registration_Date',
             'Prospect_Renewal_Date', 'Closed_Date', 'Last_Renewal']
for col in date_cols:
    if col in billings.columns:
        billings[col] = pd.to_datetime(billings[col],
                                       format='%d-%m-%Y', errors='coerce')
 
# Step 3 — Fix data types
for col in ['Proforma_Auto_Renewal', 'Proforma_World_Pay_Token']:
    if col in billings.columns:
        billings[col] = billings[col].map(
            {True: 1, False: 0, 'TRUE': 1, 'FALSE': 0,
             'true': 1, 'false': 0, 1: 1, 0: 0}
        ).astype('Int64')
 
for col in ['Current_Auto_Renewal_Flag', 'Current_World_Pay_Token']:
    if col in billings.columns:
        billings[col] = billings[col].str.strip().str.lower().map(
            {'y': 1, 'n': 0}
        ).astype('Int64')
 
if 'Proforma_Account_Stage' in billings.columns:
    billings['Proforma_Account_Stage'] = (
        billings['Proforma_Account_Stage'].str.strip().str.title()
    )
for col in ['Band', 'Last_Band', 'Tenure_Group', 'Connection_Group', 'Anchor_Group']:
    if col in billings.columns:
        billings[col] = billings[col].astype(str).str.strip()
 
# Step 4 — Fix data errors
bad_year = billings['Renewal_Year'] == 2050
if bad_year.sum() > 0:
    billings.loc[bad_year, 'Renewal_Year'] = (
        billings.loc[bad_year, 'Prospect_Renewal_Date'].dt.year
    )
mask = billings['Tenure_Years'].isna() & billings['Registration_Date'].notna()
if mask.sum() > 0:
    billings.loc[mask, 'Tenure_Years'] = (
        (billings.loc[mask, 'Prospect_Renewal_Date'] -
         billings.loc[mask, 'Registration_Date']).dt.days / 365.25
    ).round(1)

In [6]:
# Inference: keep ONLY Open rows, no churn_label
df_open = billings[billings['Prospect_Outcome'] == 'Open'].copy()
print(f"\nOpen rows after filtering: {df_open.shape}")
 
# Step 6 — Prospect_Status_Group (same map as training)
status_group_map = {
    'Renewed': 'Retained', 'Application and Money In': 'Retained',
    'Paid': 'Retained', 'Payment En Route': 'Retained_Pending',
    'Intention to Proceed': 'Retained_Pending',
    'Intention to Proceed (including auto email)': 'Retained_Pending',
    'Contact Made Deciding': 'At_Risk_Engaging',
    'Need Time To Consider < 30 days': 'At_Risk_Engaging',
    'Need time to consider': 'At_Risk_Engaging',
    'Promised to Pay - Fees not Received': 'At_Risk_Engaging',
    'Invoiced': 'At_Risk_Engaging', 'Part Payment': 'At_Risk_Engaging',
    'Potential Lost Account': 'At_Risk_Engaging',
    'Refused to Discuss': 'At_Risk_Disengaging',
    'Non Responsive': 'At_Risk_Disengaging',
    'Attempted Contact': 'At_Risk_Disengaging',
    'Contact Made': 'At_Risk_Disengaging',
    'Need Time To Consider > 30 days': 'At_Risk_Disengaging',
    'Need time to consider - 60 days': 'At_Risk_Disengaging',
    'Not to be contacted': 'At_Risk_Disengaging',
    'Renewal Proforma Sent': 'At_Risk_Disengaging',
    'No Longer Trading': 'Churned_Business',
    'Do Not Work for Client': 'Churned_Business',
    'Competitor Accreditation': 'Churned_Competitor',
    'Not Value for Money': 'Churned_Price',
    'Not Affordable': 'Churned_Price',
    'Cannot Pass Audit': 'Churned_Accreditation',
    'Not Interested': 'Churned_Disengaged',
}
df_open['Prospect_Status_Group'] = (
    df_open['Prospect_Status'].map(status_group_map).fillna('Unknown')
)
 
# Step 7 — year_key
df_open['year_key'] = (
    df_open['Co_Ref'].astype(str) + '_' +
    df_open['Renewal_Year'].astype(str)
)
 
# Step 8 — Missing value handling (same as training)
for col in ['Connection_Net', 'Connection_Qty',
            'Starting_Connection_Net', 'Starting_Connection_Qty']:
    if col in df_open.columns:
        df_open[col] = df_open[col].fillna(0)
if 'Discount_Amount' in df_open.columns:
    df_open['Discount_Amount'] = df_open['Discount_Amount'].fillna(0)
 
df_open['is_new_customer'] = df_open['Last_Years_Price'].isna().astype(int)
df_open['Last_Years_Price'] = df_open['Last_Years_Price'].fillna(df_open['Amount'])
df_open['payment_timeframe_missing'] = df_open['Payment_Timeframe'].isna().astype(int)
df_open['Payment_Timeframe'] = df_open['Payment_Timeframe'].fillna(0)
df_open['Last_Band'] = df_open['Last_Band'].fillna('New_Customer')
df_open['Last_Connections'] = df_open['Last_Connections'].fillna(0)
df_open['Last_Total_Net_Paid'] = df_open['Last_Total_Net_Paid'].fillna(0)
 
for proforma_col, current_col in [
    ('Proforma_Auto_Renewal',    'Current_Auto_Renewal_Flag'),
    ('Proforma_World_Pay_Token', 'Current_World_Pay_Token'),
]:
    if proforma_col in df_open.columns and current_col in df_open.columns:
        mask = df_open[proforma_col].isna()
        df_open.loc[mask, proforma_col] = df_open.loc[mask, current_col]
 
# Step 9 — Audit Status Group
audit_status_map = {
    'Accredited': 'Accredited',
    'Accredited - Ssip Due To Expire': 'Accredited',
    'Accred Due To Expire': 'Accredited',
    'Renewal Questionnaire Sent': 'Renewal_In_Progress',
    'Renewal Questionnaire Received': 'Renewal_In_Progress',
    '1St Renewal Questionnaire Reminder': 'Renewal_In_Progress',
    'Final Renewal Questionnaire Reminder': 'Renewal_In_Progress',
    'Renewal Report Sent - Awaiting Information': 'Renewal_In_Progress',
    'Renewal Additional Information Received': 'Renewal_In_Progress',
    '1St Reminder - Renewal Additional Info': 'Renewal_In_Progress',
    'Final Reminder - Renewal Additional Info': 'Renewal_In_Progress',
    'Failed- Renewal Questionnaire Not Received': 'Renewal_Failed',
    'Failed- Renewal Additional Information Not Received': 'Renewal_Failed',
    'Initial Questionnaire Sent': 'Initial_In_Progress',
    'Initial Questionnaire Received': 'Initial_In_Progress',
    'Initial - 1St Questionnaire Reminder Sent': 'Initial_In_Progress',
    '1St Reminder Initial Add Info': 'Initial_In_Progress',
    'Final Reminder (Initial) Additional Information Not Sent': 'Initial_In_Progress',
    'Report Sent - Awaiting Information': 'Initial_In_Progress',
    'Additional Information Received': 'Initial_In_Progress',
    'Failed- Initial Questionnaire Not Received': 'Initial_Failed',
    'Failed- Initial Additional Info Not Received': 'Initial_Failed',
    'Failed- Accreditation Removed Ssip Exp': 'Suspended_Failed',
    'Suspended - Ssip Expired': 'Suspended_Failed',
    'Suspended - 1St Reminder Ssip Expired': 'Suspended_Failed',
    'Suspended - 2Nd Reminder Ssip Expired': 'Suspended_Failed',
    'Ssip Expired - 1St Reminder': 'Suspended_Failed',
    'Suspended - Insurance Out-Of-Date': 'Suspended_Failed',
    'Withdrawn From Scheme': 'Withdrawn',
    'Unknown': 'Unknown',
}
if 'Proforma_Audit_Status' in df_open.columns:
    df_open['Audit_Status_Group'] = (
        df_open['Proforma_Audit_Status']
        .str.strip().str.title()
        .map(audit_status_map)
        .fillna('Unknown')
    )


Open rows after filtering: (8188, 56)


In [11]:
#Merging Strategy

NON_SIGNAL = {
    'Not Mentioned', 'Not Discussed', 'Not Applicable',
    'No_Interaction', 'UNAVAILABLE', 'XXXX', '', 'nan'
}
 
def meaningful_mode(s):
    meaningful = s[~s.astype(str).isin(NON_SIGNAL)].dropna()
    if len(meaningful) == 0:
        return 'Not Mentioned'
    return meaningful.mode().iloc[0]
 
def last_value(s):
    s_clean = s.dropna()
    return s_clean.iloc[-1] if len(s_clean) > 0 else np.nan
 
def binary_explode(df, col, prefix, year_key_col='year_key'):
    df_clean = df[[year_key_col, col]].copy()
    df_clean = df_clean[~df_clean[col].astype(str).isin(NON_SIGNAL)]
    df_clean = df_clean.dropna(subset=[col])
    if len(df_clean) == 0:
        return pd.DataFrame({year_key_col: df[year_key_col].unique()})
    dummies = pd.get_dummies(df_clean[col], prefix=prefix)
    dummies[year_key_col] = df_clean[year_key_col].values
    result = dummies.groupby(year_key_col).max().reset_index()
    result.columns = [
        c.lower().replace(' ', '_').replace('/', '_').replace('-', '_')
        if c != year_key_col else c
        for c in result.columns
    ]
    return result
 
# year_key for interaction tables
rc['year_key'] = (
    rc['Co_Ref'].astype(str) + '_' +
    rc['Call_Date'].astype(str)
)
cc['Call_Date'] = pd.to_datetime(cc['Call_Date'], errors='coerce')
cc['year_key'] = (
    cc['Co_Ref'].astype(str) + '_' +
    cc['Call_Date'].astype(str)
)
emails['year_key'] = (
    emails['Co_Ref'].astype(str) + '_' +
    emails['year'].astype(str)
)

In [12]:
#Renewal Calls Aggregation
binary_cols_rc = [
    'Serious_Complaint', 'Other_Complaint', 'Discussion_on_Price_Increase',
    'Renewal_Impact_Due_to_Price_Increase', 'Discount_or_Waiver_Requested',
    'Call_Reschedule_Request', 'Agent_Flagged_Membership_Status_Alert',
    'Agent_Renewal_Initiation', 'Explicit_Competitor_Mention',
    'Explicit_Switching_Intent', 'Mentioned_Competitors',
    'Price_Switching_Mentioned', 'Customer_Asked_For_Justification',
    'Discount_Offered', 'Membership_Renewal_Decision',
    'Percentage_Price_Increase_Mentioned', 'Monetary_Price_Increase_Mentioned',
]
for col in binary_cols_rc:
    if col in rc.columns:
        rc[col] = rc[col].map({'Yes': 1, 'No': 0, 'yes': 1, 'no': 0})
 
high_card_cats_rc = [
    ('Churn_Category',          'rc_churn_cat'),
    ('Complaint_Category',      'rc_complaint_cat'),
    ('Customer_Reaction_Category', 'rc_reaction_cat'),
    ('Agent_Renewal_Pitch_Category', 'rc_pitch_cat'),
    ('Customer_Renewal_Response_Category', 'rc_response_cat'),
    ('Agent_Response_Category', 'rc_agent_response_cat'),
    ('Desire_To_Cancel',        'rc_desire_cancel'),
    ('Justification_Category',  'rc_justification_cat'),
    ('Reason_For_Renewal_Category', 'rc_renewal_reason_cat'),
    ('Agent_Response_To_Cancel_Category', 'rc_cancel_response_cat'),
    ('Argument_That_Convinced_Customer_to_Stay_Category', 'rc_stay_arg_cat'),
]
low_card_cats_rc = ['Call_Direction', 'Customer_Response', 'Competitor_Value_Comparison']
 
agg_rc = {'Call_Date': 'count'}
for col in binary_cols_rc:
    if col in rc.columns:
        agg_rc[col] = 'max'
for col in low_card_cats_rc:
    if col in rc.columns:
        agg_rc[col] = meaningful_mode
 
rc = rc.sort_values(['year_key', 'Call_Date'])
rc_agg = rc.groupby('year_key').agg(agg_rc).reset_index()
rc_agg.rename(columns={'Call_Date': 'rc_total_calls'}, inplace=True)
rc_agg.columns = [
    'year_key' if c == 'year_key'
    else c if c == 'rc_total_calls'
    else f'rc_{c.lower()}' if c in low_card_cats_rc
    else f'rc_{c}'
    for c in rc_agg.columns
]
for col, prefix in high_card_cats_rc:
    if col in rc.columns:
        exploded = binary_explode(rc, col, prefix)
        rc_agg = rc_agg.merge(exploded, on='year_key', how='left')

In [14]:
# =============================================================================
# STEP 3: CC CALLS — AGGREGATE
# =============================================================================
print("Aggregating CC Calls...")

# 1. Sort by Call_Date to ensure 'last_value' is chronologically the latest
cc = cc.sort_values(['year_key', 'Call_Date'])

# 2. Map Binary Columns
binary_cols_cc = [
    'cc_care_package_discussed', 'cc_urgency_getting_on_site',
    'cc_external_consultant', 'cc_agent_cross_sell_attempt',
    'cc_customer_issues_concerns', 'cc_business_struggles_financial_hardship',
    'cc_questionnaire_completion', 'cc_chasing_response',
    'cc_issues_within_questionnaire', 'cc_login_issues',
    'cc_platform_issues', 'cc_dissatisfaction_time_to_complete',
    'cc_process_complexity_concerns', 'cc_questions_harder_than_expected',
    'cc_dissatisfaction_support', 'cc_pricing_mentioned',
    'cc_pricing_sentiment_impact', 'cc_refund_discussed',
    'cc_contractor_suggest_leave', 'cc_contractor_complained',
]

mapping_dict = {True: 1, False: 0, 'Yes': 1, 'No': 0, 'yes': 1, 'no': 0}
for col in binary_cols_cc:
    if col in cc.columns:
        cc[col] = cc[col].map(mapping_dict)

# 3. Numeric Conversion
numeric_cols_cc = [
    'cc_contractor_sentiment_start_score',
    'cc_contractor_sentiment_end_score',
    'cc_contractor_sentiment_overall_score',
    'cc_contractor_sentiment_issues_score',
    'days_to_renewal',
]
for col in numeric_cols_cc:
    if col in cc.columns:
        cc[col] = pd.to_numeric(cc[col], errors='coerce')

# 4. Define Aggregation Map
# Use a copy to avoid dictionary mutation issues
agg_cc = {'Call_Date': 'count'}

for col in binary_cols_cc:
    if col in cc.columns:
        agg_cc[col] = 'max'

for col in numeric_cols_cc:
    if col in cc.columns:
        # Note: using a list creates a MultiIndex header
        agg_cc[col] = ['mean', 'min', last_value]

low_card_cats_cc = ['Direction', 'cc_call_initiated_by']
for col in low_card_cats_cc:
    if col in cc.columns:
        # Assuming meaningful_mode is defined elsewhere in your script
        agg_cc[col] = meaningful_mode

# 5. Perform Aggregation
# We keep year_key in the index during flattening to protect it
cc_agg = cc.groupby('year_key').agg(agg_cc)

# 6. Flatten MultiIndex Columns
# This logic checks if a column has sub-levels (like 'mean', 'min') and joins them
new_cols = []
for col_name, agg_func in cc_agg.columns:
    if agg_func == '' or pd.isna(agg_func) or agg_func == 'meaningful_mode':
        new_cols.append(col_name)
    else:
        # Convert function objects like <function last_value> to a string
        func_str = agg_func.__name__ if hasattr(agg_func, '__name__') else str(agg_func)
        new_cols.append(f"{col_name}_{func_str}")

cc_agg.columns = new_cols

# 7. Clean up column names and bring year_key back
cc_agg = cc_agg.reset_index() # year_key is now safely a column
cc_agg.columns = [c.replace('<lambda>', 'last').replace('last_value', 'last') for c in cc_agg.columns]
cc_agg.rename(columns={'Call_Date_count': 'cc_total_calls'}, inplace=True)

# 8. Merge High-Cardinality "Exploded" Columns
high_card_cats_cc = [
    ('cc_contractor_sentiment', 'cc_sentiment'),
    ('cc_care_package',         'cc_package'),
]

for col, prefix in high_card_cats_cc:
    if col in cc.columns:
        exploded = binary_explode(cc, col, prefix)
        # Verify year_key exists in both before merging
        if 'year_key' in exploded.columns and 'year_key' in cc_agg.columns:
            cc_agg = cc_agg.merge(exploded, on='year_key', how='left')
        else:
            print(f"Warning: Could not merge {col}. year_key missing.")

print(f"CC Calls aggregated      : {cc_agg.shape}")

Aggregating CC Calls...
CC Calls aggregated      : (2024, 45)


In [15]:
# Emails Aggregation
binary_cols_em = [
    'crm_accreditation_completed', 'crm_timely_completion',
    'crm_progress_towards_accreditation', 'crm_delays_in_accreditation',
    'crm_contractor_suggested_leave', 'crm_contractor_engagement',
    'crm_dts_or_ssip_mentioned', 'crm_customer_payment_intention',
    'crm_competitors_mentioned', 'crm_platform_issues_raised',
    'crm_agent_chased_contractor', 'crm_accreditation_issues',
    'crm_membership_overdue', 'crm_dissatisified_with_renewal_price',
    'crm_customer_complained', 'crm_refund_mentioned',
    'crm_negative_customer_experience', 'crm_dissatisfaction_with_support',
    'crm_financial_hardship_mentioned',
]
for col in binary_cols_em:
    if col in emails.columns:
        emails[col] = emails[col].map(
            {'Yes': 1, 'No': 0, 'Not Discussed': 0, 'yes': 1, 'no': 0}
        )
 
numeric_cols_em = ['crm_contractor_sentiment_score', 'crm_agent_chase_count']
for col in numeric_cols_em:
    if col in emails.columns:
        emails[col] = pd.to_numeric(emails[col], errors='coerce')
 
cat_cols_em = ['crm_contractor_sentiment', 'crm_membership_level', 'crm_auto_renewal_status']
 
agg_em = {'Time_to_Renewal': 'count'}
for col in binary_cols_em:
    if col in emails.columns:
        agg_em[col] = 'max'
for col in numeric_cols_em:
    if col in emails.columns:
        agg_em[col] = 'mean'
for col in cat_cols_em:
    if col in emails.columns:
        agg_em[col] = meaningful_mode
 
em_agg = emails.groupby('year_key').agg(agg_em).reset_index()
em_agg.rename(columns={'Time_to_Renewal': 'em_total_emails'}, inplace=True)
 
high_card_cats_em = [
    ('crm_contractor_sentiment', 'em_sentiment'),
    ('crm_membership_level', 'em_membership_level'),
]
for col, prefix in high_card_cats_em:
    if col in emails.columns:
        exploded = binary_explode(emails, col, prefix)
        em_agg = em_agg.merge(exploded, on='year_key', how='left')

In [16]:
# ── Final merge onto Open rows ────────────────────────────────────────────────
df_open = df_open.merge(rc_agg, on='year_key', how='left')
df_open = df_open.merge(cc_agg, on='year_key', how='left')
df_open = df_open.merge(em_agg, on='year_key', how='left')
print(f"\nAfter merge: {df_open.shape}")
 
# Post-merge null fill (same as training)
for col in ['rc_total_calls', 'cc_total_calls', 'em_total_emails']:
    if col in df_open.columns:
        df_open[col] = df_open[col].fillna(0)
 
for col in df_open.columns:
    if df_open[col].dtype in ['float64', 'int64', 'Int64']:
        if col.startswith(('rc_', 'cc_', 'crm_', 'em_')):
            df_open[col] = df_open[col].fillna(0)
 
for col in df_open.columns:
    if df_open[col].dtype == object:
        if col.startswith(('rc_', 'cc_', 'crm_', 'em_')):
            df_open[col] = df_open[col].fillna('No_Interaction')


After merge: (8188, 348)


In [17]:
# =============================================================================
# ENGINEER SAME DERIVED FEATURES AS TRAINING
# =============================================================================
df_open['price_change'] = df_open['Amount'] - df_open['Last_Years_Price']
df_open['price_increase_pct'] = (
    df_open['price_change'] / df_open['Last_Years_Price'].replace(0, np.nan) * 100
).round(2).fillna(0)
 
# proforma_delay (same as merge.ipynb Cell 15)
df_open['Prospect_Renewal_Date'] = pd.to_datetime(
    df_open['Prospect_Renewal_Date'], errors='coerce'
)
df_open['Proforma_Date'] = pd.to_datetime(
    df_open['Proforma_Date'], errors='coerce'
)
df_open['proforma_delay'] = (
    df_open['Prospect_Renewal_Date'] - df_open['Proforma_Date']
).dt.days
 
# had_*_interaction flags
df_open['had_rc_interaction'] = (df_open['rc_total_calls'] > 0).astype(int)
df_open['had_cc_interaction'] = (df_open['cc_total_calls'] > 0).astype(int)
df_open['had_em_interaction'] = (df_open['em_total_emails'] > 0).astype(int)
 
# payment flags
df_open['payment_is_bacs']     = (df_open['Payment_Method'] == 'BACS').astype(int)
df_open['payment_is_card']     = (df_open['Payment_Method'] == 'CARD').astype(int)
df_open['payment_is_worldpay'] = (df_open['Payment_Method'] == 'WORLD PAY').astype(int)
df_open['payment_unknown']     = (df_open['Payment_Method'] == 'UNKNOWN').astype(int)
 
# em_sentiment_dissatisfied flag
sent_col = 'em_sentiment_dissatisfied_true'
if sent_col not in df_open.columns:
    df_open[sent_col] = 0

C:\Users\pragi\AppData\Local\Temp\ipykernel_9244\737759428.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_open['price_change'] = df_open['Amount'] - df_open['Last_Years_Price']
C:\Users\pragi\AppData\Local\Temp\ipykernel_9244\737759428.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_open['price_increase_pct'] = (
C:\Users\pragi\AppData\Local\Temp\ipykernel_9244\737759428.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perfo

In [18]:
# =============================================================================
# SELECT EXACTLY THE 49 FEATURES USED IN TRAINING
# =============================================================================
FEATURE_COLS = [
    'Total_Renewal_Score_New', 'Sustainability_Score', 'Auto_Renewal_Score',
    'Tenure_Scores', 'Current_World_Pay_Token', 'Renewal_Score_At_Release',
    'Proforma_Membership_Status', 'Discount_Amount', 'price_increase_pct',
    'Gross', 'Proforma_Account_Stage', 'Audit_Status_Group',
    'Current_Anchorings', 'Last_Band', 'price_change', 'Anchoring_Score',
    'Proforma_World_Pay_Token', 'Tenure_Years', '#_of_Connection',
    'is_new_customer', 'Last_Connections', 'Tenure_Group',
    'rc_desire_cancel_desired_to_cancel_true', 'Current_Auto_Renewal_Flag',
    'Amount', 'PQQNet', 'had_rc_interaction', 'rc_Membership_Renewal_Decision',
    'Band', 'had_em_interaction', 'Connection_Group', 'crm_agent_chase_count',
    'crm_negative_customer_experience', 'Anchor_Group',
    'crm_financial_hardship_mentioned',
    'rc_agent_response_cat_cancellation___termination___closure_true',
    'crm_contractor_engagement', 'crm_membership_overdue',
    'Proforma_Auto_Renewal', 'crm_customer_payment_intention',
    'rc_response_cat_cancellation___termination___non_renewal___withdrawal_true',
    'crm_progress_towards_accreditation', 'crm_customer_complained',
    'Connection_Qty', 'crm_dissatisified_with_renewal_price',
    'rc_churn_cat_cost___price___too_expensive_true',
    'rc_churn_cat_financial_struggles___business_problems_true',
    'had_cc_interaction',
]
 
ID_COLS = ['Co_Ref', 'year_key', 'Renewal_Year']
 
# Add any missing feature columns as 0
for col in FEATURE_COLS:
    if col not in df_open.columns:
        df_open[col] = 0
        print(f"  Added missing column as 0: {col}")
 
# Extract Co_Ref from year_key
df_open['Co_Ref'] = df_open['year_key'].str.rsplit('_', n=1).str[0]
 
# Build inference dataframe
df_inference = df_open[ID_COLS + FEATURE_COLS].copy()
 
print("\n" + "=" * 60)
print("INFERENCE DATASET — OPEN ROWS")
print("=" * 60)
print(f"Shape    : {df_inference.shape}")
print(f"Rows     : {len(df_inference):,} open renewal cycles")
print(f"Features : {len(FEATURE_COLS)}")
 
missing_check = df_inference[FEATURE_COLS].isnull().sum()
missing_check = missing_check[missing_check > 0]
if len(missing_check) > 0:
    print(f"\nNulls remaining:")
    print(missing_check)
else:
    print(f"\nNo missing values in features.")

  Added missing column as 0: rc_desire_cancel_desired_to_cancel_true
  Added missing column as 0: rc_agent_response_cat_cancellation___termination___closure_true
  Added missing column as 0: rc_response_cat_cancellation___termination___non_renewal___withdrawal_true
  Added missing column as 0: rc_churn_cat_cost___price___too_expensive_true
  Added missing column as 0: rc_churn_cat_financial_struggles___business_problems_true

INFERENCE DATASET — OPEN ROWS
Shape    : (8188, 51)
Rows     : 8,188 open renewal cycles
Features : 48

Nulls remaining:
Renewal_Score_At_Release       4
Proforma_Membership_Status     4
Proforma_Account_Stage         4
Tenure_Years                  43
#_of_Connection                4
dtype: int64


C:\Users\pragi\AppData\Local\Temp\ipykernel_9244\326548900.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_open[col] = 0
C:\Users\pragi\AppData\Local\Temp\ipykernel_9244\326548900.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_open[col] = 0
C:\Users\pragi\AppData\Local\Temp\ipykernel_9244\326548900.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) in

In [ ]:
df_inf = df_inference.copy()
for col in FEATURE_COLS:
    if col not in df_inf.columns:
        df_inf[col] = 0

num_cols = df_inf.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df_inf.select_dtypes(include=['object']).columns

for col in num_cols:
    if col in df_inf.columns:
        median_val = df_inf[col].median()
        df_inf[col] = df_inf[col].fillna(median_val)
        
for col in cat_cols:
    if col in df_inf.columns:
        mode_val = df_inf[col].mode()[0]
        df_inf[col] = df_inf[col].fillna(mode_val)

df_inf[FEATURE_COLS] = df_inf[FEATURE_COLS].fillna(0)

df_inf = df_inf[ID_COLS + FEATURE_COLS]

nulls = df_inf[FEATURE_COLS].isnull().sum()
nulls = nulls[nulls > 0]

if len(nulls) == 0:
    print("✅ No nulls remaining")
else:
    print("⚠️ Still nulls found:")
    print(nulls)

✅ No nulls remaining


In [22]:
import pandas as pd
import numpy as np

# 1. Load the data (assuming the raw text you provided is stored in 'inference_df')
# Using the column list provided in your prompt
inference_cols = [
    "Co_Ref","year_key","Renewal_Year","Total_Renewal_Score_New","Sustainability_Score",
    "Auto_Renewal_Score","Tenure_Scores","Current_World_Pay_Token","Renewal_Score_At_Release",
    "Proforma_Membership_Status","Discount_Amount","price_increase_pct","Gross",
    "Proforma_Account_Stage","Audit_Status_Group","Current_Anchorings","Last_Band",
    "price_change","Anchoring_Score","Proforma_World_Pay_Token","Tenure_Years",
    "#_of_Connection","is_new_customer","Last_Connections","Tenure_Group",
    "rc_desire_cancel_desired_to_cancel_true","Current_Auto_Renewal_Flag","Amount",
    "PQQNet","had_rc_interaction","rc_Membership_Renewal_Decision","Band",
    "had_em_interaction","Connection_Group","crm_agent_chase_count",
    "crm_negative_customer_experience","Anchor_Group","crm_financial_hardship_mentioned",
    "rc_agent_response_cat_cancellation___termination___closure_true",
    "crm_contractor_engagement","crm_membership_overdue","Proforma_Auto_Renewal",
    "crm_customer_payment_intention","rc_response_cat_cancellation___termination___non_renewal___withdrawal_true",
    "crm_progress_towards_accreditation","crm_customer_complained","Connection_Qty",
    "crm_dissatisified_with_renewal_price","rc_churn_cat_cost___price___too_expensive_true",
    "rc_churn_cat_financial_struggles___business_problems_true","had_cc_interaction"
]

# 2. Value Handling & Data Cleaning
def clean_inference_data(df):
    temp_df = df.copy()
    
    # Handle Numeric cleaning (remove % and handle nans)
    if 'price_increase_pct' in temp_df.columns:
        temp_df['price_increase_pct'] = temp_df['price_increase_pct'].astype(str).str.replace('%', '')
        temp_df['price_increase_pct'] = pd.to_numeric(temp_df['price_increase_pct'], errors='coerce').fillna(0)
    
    # Fill numeric NAs with 0 for logic consistency
    numeric_cols = temp_df.select_dtypes(include=[np.number]).columns
    temp_df[numeric_cols] = temp_df[numeric_cols].fillna(0)
    
    # Fill categorical NAs
    cat_cols = temp_df.select_dtypes(include=['object']).columns
    temp_df[cat_cols] = temp_df[cat_cols].fillna('Unknown')
    
    return temp_df

# 3. Schema Transformation (Mapping Inference -> Final Prediction Format)
def transform_to_final_prediction(df):
    final_df = clean_inference_data(df)

    
    # Logic based transformation (Example: mapping churn_label)
    # Since this is an inference set, these are usually generated by your model
    # Here we add placeholders to match the requested "Full Prediction Level" structure
    final_df['churn_label'] = 0 
    final_df['renewal_month_year'] = "Pending"
    
    # These would normally come from model.predict_proba() and model.predict()
    final_df['churn_probability'] = 0.0  
    final_df['churn_predicted'] = 0
    final_df['churn_predicted_label'] = "Won"
    
    # Reorder columns to match the "final_predictions" structure provided
    final_cols_order = [
        "Co_Ref","year_key","Renewal_Year",
        "Total_Renewal_Score_New",
        "Sustainability_Score","Auto_Renewal_Score",
        "Tenure_Scores","Current_World_Pay_Token",
        "Renewal_Score_At_Release","Proforma_Membership_Status","Discount_Amount",
        "price_increase_pct","Gross","Proforma_Account_Stage","Audit_Status_Group",
        "Current_Anchorings","Last_Band","price_change","Anchoring_Score",
        "Proforma_World_Pay_Token","Tenure_Years","#_of_Connection","is_new_customer",
        "Last_Connections","Tenure_Group","rc_desire_cancel_desired_to_cancel_true",
        "crm_contractor_suggested_leave","Current_Auto_Renewal_Flag",
        "crm_contractor_sentiment_score","Amount","PQQNet","had_rc_interaction",
        "rc_Membership_Renewal_Decision","Band",
        "had_em_interaction","Connection_Group","crm_agent_chase_count",
        "crm_negative_customer_experience","Anchor_Group","crm_financial_hardship_mentioned",
        "rc_agent_response_cat_cancellation___termination___closure_true",
        "crm_contractor_engagement","crm_membership_overdue","Proforma_Auto_Renewal",
        "crm_customer_payment_intention","rc_response_cat_cancellation___termination___non_renewal___withdrawal_true",
        "crm_progress_towards_accreditation","crm_customer_complained","Connection_Qty",
        "crm_dissatisified_with_renewal_price","rc_churn_cat_cost___price___too_expensive_true",
        "rc_churn_cat_financial_struggles___business_problems_true","had_cc_interaction",
        "churn_label","renewal_month_year","churn_probability","churn_predicted","churn_predicted_label"
    ]
    
    # Add any columns that are in final_cols_order but missing in final_df as 0s/Nones
    for col in final_cols_order:
        if col not in final_df.columns:
            final_df[col] = 0
            
    return final_df[final_cols_order]

# Execute
final_predictions_df = transform_to_final_prediction(df_inference)

print("Transformation Complete. Shape:", final_predictions_df.shape)
display(final_predictions_df.head())

Transformation Complete. Shape: (8188, 58)


,Co_Ref,year_key,Renewal_Year,Total_Renewal_Score_New,Sustainability_Score,Auto_Renewal_Score,Tenure_Scores,Current_World_Pay_Token,Renewal_Score_At_Release,Proforma_Membership_Status,...,Connection_Qty,crm_dissatisified_with_renewal_price,rc_churn_cat_cost___price___too_expensive_true,rc_churn_cat_financial_struggles___business_problems_true,had_cc_interaction,churn_label,renewal_month_year,churn_probability,churn_predicted,churn_predicted_label
0,FL6837,FL6837_2026,2026,43.0,9.5,8,8.0,0,26.0,Accredited,...,0.0,0.0,0,0,0,0,Pending,0.0,0,Won
1,HG6294,HG6294_2026,2026,44.0,9.5,8,9.5,0,27.5,Accredited,...,0.0,0.0,0,0,0,0,Pending,0.0,0,Won
2,TJ2968,TJ2968_2026,2026,45.5,9.5,8,9.5,0,28.0,Accredited,...,0.0,0.0,0,0,0,0,Pending,0.0,0,Won
3,SS4827,SS4827_2026,2026,43.5,8.0,8,9.5,0,27.5,Accredited,...,0.0,0.0,0,0,0,0,Pending,0.0,0,Won
4,LE8497,LE8497_2026,2026,45.0,9.5,8,9.5,0,27.5,Accredited,...,0.0,0.0,0,0,0,0,Pending,0.0,0,Won


In [23]:
# Assuming df_open still contains the 'Prospect_Renewal_Date' column which was converted to datetime earlier
# We map this back to the final dataframe using the year_key

# 1. Create a mapping Series from the original open rows
renewal_date_map = df_open.set_index('year_key')['Prospect_Renewal_Date']

# 2. Inside your transformation logic, apply the formatting:
# This converts '2025-01-15' into 'Jan 2025'
final_predictions_df['renewal_month_year'] = final_predictions_df['year_key'].map(renewal_date_map).dt.strftime('%b %Y')

In [24]:
print("Transformation Complete. Shape:", final_predictions_df.shape)
display(final_predictions_df.head())

Transformation Complete. Shape: (8188, 58)


,Co_Ref,year_key,Renewal_Year,Total_Renewal_Score_New,Sustainability_Score,Auto_Renewal_Score,Tenure_Scores,Current_World_Pay_Token,Renewal_Score_At_Release,Proforma_Membership_Status,...,Connection_Qty,crm_dissatisified_with_renewal_price,rc_churn_cat_cost___price___too_expensive_true,rc_churn_cat_financial_struggles___business_problems_true,had_cc_interaction,churn_label,renewal_month_year,churn_probability,churn_predicted,churn_predicted_label
0,FL6837,FL6837_2026,2026,43.0,9.5,8,8.0,0,26.0,Accredited,...,0.0,0.0,0,0,0,0,May 2026,0.0,0,Won
1,HG6294,HG6294_2026,2026,44.0,9.5,8,9.5,0,27.5,Accredited,...,0.0,0.0,0,0,0,0,May 2026,0.0,0,Won
2,TJ2968,TJ2968_2026,2026,45.5,9.5,8,9.5,0,28.0,Accredited,...,0.0,0.0,0,0,0,0,Apr 2026,0.0,0,Won
3,SS4827,SS4827_2026,2026,43.5,8.0,8,9.5,0,27.5,Accredited,...,0.0,0.0,0,0,0,0,Mar 2026,0.0,0,Won
4,LE8497,LE8497_2026,2026,45.0,9.5,8,9.5,0,27.5,Accredited,...,0.0,0.0,0,0,0,0,May 2026,0.0,0,Won


In [25]:
# Save final inference dataset
final_predictions_df.to_csv("dataset/inference/inference_dataset.csv", index=False)